In [ ]:
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt 
import numpy as np
import os
from evcouplings import align

In [ ]:
dms_df = pd.read_csv('/n/groups/marks/projects/viral_families/priority-viruses/data/reference_files/viral_dms_reference.csv')
evh_spearman = pd.read_csv("/n/groups/marks/projects/viral_families/priority-viruses/results/summary/ali_selection_summary.csv")

In [ ]:
theta = 0.99
min_cov= 0.5

def read_single_evh_file(dms_id, base_dir, bitscore, database):
    """
    Given a row with EVH metadata, load the corresponding single_mutant_matrix.csv.
    """
    # Construct prefix
    theta = 0.99
    seq_cov = 50
    col_cov = 50
    prefix = f"{dms_id}_seqcov{seq_cov}_colcov{col_cov}_theta{theta}"

    # Format bitscore for filename
    bitscore_tag = f"b.{str(bitscore).split('.')[-1]}"

    # Build full file path
    file_path = os.path.join(
        base_dir,
        "output",
        database,
        prefix,
        f"{prefix}_{bitscore_tag}/align/{prefix}_{bitscore_tag}.a2m"
    )


    return file_path


def read_weights_file(dms_id, base_dir, bitscore, database):
    """
    Given a row with EVH metadata, load the corresponding single_mutant_matrix.csv.
    """
    # Construct prefix
    theta = 0.99
    seq_cov = 50
    col_cov = 50
    prefix = f"{dms_id}_seqcov{seq_cov}_colcov{col_cov}_theta{theta}"

    # Format bitscore for filename
    bitscore_tag = f"b.{str(bitscore).split('.')[-1]}"

    # Build full file path
    file_path = os.path.join(
        base_dir,
        "output",
        database,
        prefix,
        f"{prefix}_{bitscore_tag}/align/{prefix}_{bitscore_tag}_inverse_sequence_weights.csv"
    )
    return file_path

#path with EVCouplings alignments
base_evh_path = "/n/groups/marks/projects/viral_families/models/evh"
dbs = ["uniref100", "uniref90", "uniref_bfd_mgnify"]
bitscores = ["b.03","b.04","b.05","b.1","b.3","b.5"]


In [6]:
evh_spearman

,DMS ID,EVCouplings Spearman,Bitscore,Database,len_seq,Virus,assay,log_neff90,Num_Seq,Prop_90,Len_Seq,Neff90
0,AAV2_CAPSD_Sinai,0.474905,0.50,uniref100,735,AAV2,fitness,0.005201,1215,0.106173,735,45.745523
1,AAV2_CAPSD_Sinai,0.491122,0.30,uniref100,735,AAV2,fitness,0.005149,3563,0.036486,735,44.012486
2,AAV2_CAPSD_Sinai,0.445024,0.10,uniref100,735,AAV2,fitness,0.005767,3552,0.051520,735,69.319327
3,AAV2_CAPSD_Sinai,0.466503,0.05,uniref100,735,AAV2,fitness,0.005030,3567,0.036726,735,40.320799
4,AAV2_CAPSD_Sinai,0.464024,0.04,uniref100,735,AAV2,fitness,0.005132,3577,0.043892,735,43.462455
...,...,...,...,...,...,...,...,...,...,...,...,...
605,BPP22_COAT_Tsuboyama,0.191842,0.30,uniref90,40,Salmonella phage P22,stability,0.017329,38858,0.000077,40,2.000000
606,BPP22_COAT_Tsuboyama,0.193267,0.10,uniref90,40,Salmonella phage P22,stability,0.017329,97594,0.000031,40,2.000000
607,BPP22_COAT_Tsuboyama,0.129609,0.05,uniref90,40,Salmonella phage P22,stability,0.017329,168979,0.000018,40,2.000000
608,BPP22_COAT_Tsuboyama,0.154978,0.04,uniref90,40,Salmonella phage P22,stability,0.044794,216592,0.000032,40,6.000000


In [ ]:
# Initialize Neff columns
for col in ["neff99","neff95","neff90","neff85","neff80","neff75",
            "neff70","neff65","neff60","neff55","neff50","neff30"]:
    merged_reference[col] = None

dms_neffs = {}

for _, row in dms_df.iterrows():
    dms_id = row["name"]

    for database in dbs:
        for bitscore in bitscores:
            msa_path = read_single_evh_file(dms_id, base_evh_path, bitscore, database)
            if not os.path.exists(msa_path):
                print("doesn't exist", dms_id, bitscore, database)
                continue

            with open(msa_path, "r") as fh:
                alignment = align.Alignment.from_file(fh)

                # Filter sequences by per-sequence coverage
                keep_mask = (1 - alignment.count("-", axis="seq")) >= min_cov
                alignment = alignment.select(sequences=keep_mask)

                # Reference sequence (first in alignment)
                ref_seq = "".join(alignment[0])

                # Load weights
                weights_path = read_weights_file(dms_id, base_evh_path, bitscore, database)
                if os.path.isfile(weights_path):
                    weights = pd.read_csv(weights_path)["num_cluster_members"].to_numpy()
                else:
                    print("no weights", dms_id)

                    continue

                # Identities to reference
                identities = alignment.identities_to(ref_seq)
                print(len(identities), len(weights))

            # Metrics
            weights = 1 / weights
            neff = np.sum(weights)
            '''
            neff_id99 = sum(w for j, w in enumerate(weights) if identities[j] > 0.99)
            neff_id95 = sum(w for j, w in enumerate(weights) if identities[j] > 0.95)
            neff_id90 = sum(w for j, w in enumerate(weights) if identities[j] > 0.90)
            neff_id85 = sum(w for j, w in enumerate(weights) if identities[j] > 0.85)
            neff_id80 = sum(w for j, w in enumerate(weights) if identities[j] > 0.80)
            neff_id75 = sum(w for j, w in enumerate(weights) if identities[j] > 0.75)
            neff_id70 = sum(w for j, w in enumerate(weights) if identities[j] > 0.70)
            neff_id65 = sum(w for j, w in enumerate(weights) if identities[j] > 0.65)
            neff_id60 = sum(w for j, w in enumerate(weights) if identities[j] > 0.60)
            neff_id55 = sum(w for j, w in enumerate(weights) if identities[j] > 0.55)
            neff_id50 = sum(w for j, w in enumerate(weights) if identities[j] > 0.50)
            neff_id30 = sum(w for j, w in enumerate(weights) if identities[j] > 0.30)
            '''
            seq_len = len(ref_seq)
            num_seq = len(identities)
            key = (dms_id, database, bitscore)
            
            #For Calculating Prop Metrics
            prop_99 = sum(1 for v in identities if v > 0.99) / num_seq
            prop_95 = sum(1 for v in identities if v > 0.95) / num_seq
            prop_90 = sum(1 for v in identities if v > 0.90) / num_seq
            prop_85 = sum(1 for v in identities if v > 0.85) / num_seq
            prop_80 = sum(1 for v in identities if v > 0.80) / num_seq
            prop_75 = sum(1 for v in identities if v > 0.75) / num_seq
            prop_70 = sum(1 for v in identities if v > 0.70) / num_seq
            prop_65 = sum(1 for v in identities if v > 0.65) / num_seq
            prop_60 = sum(1 for v in identities if v > 0.60) / num_seq
            prop_55 = sum(1 for v in identities if v > 0.55) / num_seq
            prop_50 = sum(1 for v in identities if v > 0.50) / num_seq
            prop_30 = sum(1 for v in identities if v > 0.30) / num_seq
     

            dms_neffs[key] = {
                "ID": dms_id,
                "Database": database,
                "Bitscore": bitscore,
                "Neff": neff,
                # "Neff99": neff_id99,
                # "Neff95": neff_id95,
                # "Neff90": neff_id90,
                # "Neff85": neff_id85,
                # "Neff80": neff_id80,
                # "Neff75": neff_id75,
                # "Neff70": neff_id70,
                # "Neff65": neff_id65,
                # "Neff60": neff_id60,
                # "Neff55": neff_id55,
                # "Neff50": neff_id50,
                # "Neff30": neff_id30,
                "prop99": prop_99,
                "prop95": prop_95,
                "prop90": prop_90,
                "prop85": prop_85,
                "prop80": prop_80,
                "prop75": prop_75,
                "prop70": prop_70,
                "prop65": prop_65,
                "prop60": prop_60,
                "prop55": prop_55,
                "prop50": prop_50,
                "prop30": prop_30,
                "NumSeq": num_seq,
                "SeqLen": seq_len,
                "MSAFile": msa_path,
            }
records = list(dms_neffs.values())
df_all_entries = pd.DataFrame(records)

In [11]:
df_final

,DMS ID,Database,Bitscore,prop,prop99,prop95,prop90,prop85,prop80,prop75,prop70,prop65,prop60,prop55,prop50,prop30
0,LASSA_GP_Carr,uniref100,b.03,0.346580,0.000924,0.005545,0.346580,0.523105,0.533272,0.556377,0.586876,0.617375,0.639556,0.740296,0.857671,0.925139
1,LASSA_GP_Carr,uniref100,b.04,0.405681,0.008740,0.092498,0.405681,0.417334,0.423889,0.446468,0.482884,0.489439,0.520757,0.641661,0.678806,0.781500
2,LASSA_GP_Carr,uniref100,b.05,0.490317,0.010563,0.111796,0.490317,0.504401,0.512324,0.540493,0.584507,0.591549,0.630282,0.775528,0.821303,0.988556
3,LASSA_GP_Carr,uniref100,b.1,0.490317,0.010563,0.111796,0.490317,0.504401,0.512324,0.540493,0.584507,0.591549,0.629401,0.775528,0.821303,0.989437
4,LASSA_GP_Carr,uniref100,b.3,0.490317,0.010563,0.111796,0.490317,0.504401,0.512324,0.540493,0.584507,0.591549,0.625000,0.775528,0.821303,0.986796
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
653,LAMBDA_HCP_Tsuboyama,uniref90,b.5,0.006829,0.003902,0.003902,0.006829,0.013659,0.023415,0.028293,0.031220,0.039024,0.044878,0.067317,0.108293,0.879024
654,LAMBDA_HCP_Tsuboyama,uniref_bfd_mgnify,b.5,0.045629,0.014477,0.039949,0.045629,0.070552,0.085212,0.091259,0.096207,0.102071,0.107752,0.152465,0.203775,0.886751
655,BP434_RPC1_Tsuboyama,uniref100,b.5,0.000063,0.000001,0.000056,0.000063,0.000073,0.000095,0.000117,0.000180,0.000229,0.000329,0.000554,0.001482,0.485482
656,BP434_RPC1_Tsuboyama,uniref90,b.3,0.000007,0.000002,0.000005,0.000007,0.000009,0.000017,0.000033,0.000061,0.000098,0.000211,0.000659,0.004771,0.764107
